### IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore") # Suppress warnings for cleaner output

### LOAD DATA

In [ ]:
df = pd.read_csv("../data/household_power_consumption.txt", sep=";", na_values=["?"], low_memory=False)
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
df = df.drop(columns=["Date", "Time"])
df = df.set_index("datetime").sort_index()
df.head()

hourly_power = df["Global_active_power"].resample("h").mean()
print("hourly_power_shape", hourly_power.shape)

### ANOMALY DETECTION FUNCTION

In [ ]:
def detect_anomalies(series, rolling_window = 24, spike_std = 3, flat_var = 0.001, flat_window = 6):
    """
    Detects and labels anomalies in a time series
    Returns a dataframe with original values and anomaly labels
    Labels are:
    0: normal
    1: missing/dropout
    2: spike
    3: flatline
    """

    result = pd.DataFrame({'value': series}) 
    result['label'] = 0  # Initialize all as normal
    result['anomaly_type'] = 'normal'  # Initialize all as normal

    # 1. Detect missing values (dropouts)
    mask_missing = result['value'].isna()
    result.loc[mask_missing, 'label'] = 1
    result.loc[mask_missing, 'anomaly_type'] = 'dropout'

    # 2. Detect spikes
    rolling_mean = result['value'].rolling(window=rolling_window, center = True, min_periods=1).mean()
    rolling_std = result['value'].rolling(window=rolling_window, center = True, min_periods=1).std()
    mask_spike = (abs(result['value'] - rolling_mean) > (spike_std * rolling_std)) & ~mask_missing
    result.loc[mask_spike, 'label'] = 2
    result.loc[mask_spike, 'anomaly_type'] = 'spike'

    # 3. Detect flatlines
    rolling_var = result['value'].rolling(window=flat_window, min_periods=1).var()
    mask_flat = (rolling_var < flat_var) & ~mask_missing & result['value']>0 # Only consider flatlines for non-zero values because zero values can be normal in power consumption data
    result.loc[mask_flat, 'label'] = 3
    result.loc[mask_flat, 'anomaly_type'] = 'flatline'

    return result

anomalies = detect_anomalies(hourly_power)
print(anomalies['label'].value_counts()) # value counts of each anomaly type
